In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from collections import defaultdict

# ===== 유틸 =====
def log(msg): 
    print(f"[INFO] {msg}")

def ensure_int_codes(s: pd.Series):
    """코드형 값은 가급적 int로 캐스팅. 캐스팅 불가값/NaN은 남김."""
    try:
        return pd.to_numeric(s, errors="ignore").astype("Int64")
    except Exception:
        return s

def safe_map(series, mapping):
    """딕셔너리 매핑(없는 값은 그대로 유지)"""
    return series.map(mapping).fillna(series)

def apply_map(df, col, mapping):
    if col in df.columns:
        df[col] = safe_map(df[col], mapping)
        df[col] = ensure_int_codes(df[col])
    return df

def fillna_with(df, col, value):
    if col in df.columns:
        df[col] = df[col].fillna(value)
        df[col] = ensure_int_codes(df[col])
    return df

def uppercase_col(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper()
    return df

def sum_cols(df, new_col, cols):
    existing = [c for c in cols if c in df.columns]
    if existing:
        df[new_col] = df[existing].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1).astype("Int64")
    return df

def parse_code_cell(x):
    """
    2015/2017 '참여단체_*순위코드' 등에 '1,3,5' 같은 문자열/리스트가 들어온 경우 robust 파싱
    """
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set)):
        return [int(v) for v in x if pd.notna(v)]
    s = str(x).strip()
    if not s:
        return []
    # 쉼표/공백 구분 모두 허용
    parts = [p for chunk in s.split(",") for p in chunk.strip().split()]
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except:
            # 숫자 아닌 토큰은 무시
            pass
    return out

# ===== 참가단체(더미화) =====
PARTICIPATION_ORDERED = [
    "단체참여_친목사교단체여부",
    "단체참여_종교단체여부",
    "단체참여_취미스포츠여가활동단체여부",
    "단체참여_시민사회단체여부",
    "단체참여_학술단체여부",
    "단체참여_이익단체여부",
    "단체참여_정치단체여부",
    "단체참여_지역사회모임여부",
    "단체참여_기타여부",
]

def build_participation_dummies_from_ranks(df, rank_cols):
    """
    rank_cols 예: ['참여단체_1순위코드','참여단체_2순위코드','참여단체_3순위코드']
    코드값 1~9 -> PARTICIPATION_ORDERED의 1~9번째 항목 더미로 변환(있으면 1, 없으면 0)
    """
    # 초기화
    for col in PARTICIPATION_ORDERED:
        if col not in df.columns:
            df[col] = 0

    if not all([c in df.columns for c in rank_cols]):
        return df

    # 각 행 처리
    code_to_col = {i+1: col for i, col in enumerate(PARTICIPATION_ORDERED)}  # 1..9 매핑
    def one_row(row):
        codes = []
        for rc in rank_cols:
            codes += parse_code_cell(row[rc])
        codes = set([c for c in codes if 1 <= c <= 9])
        out = {}
        for i in range(1, 10):
            col = code_to_col[i]
            out[col] = 1 if i in codes else 0
        return pd.Series(out)

    df[PARTICIPATION_ORDERED] = df.apply(one_row, axis=1)
    # 더 이상 필요 없으면 rank 컬럼 유지/삭제는 팀 규칙에 맞게 선택
    return df

def ensure_participation_binary(df):
    """2019+의 이미-이진 더미들도 빠진 게 있으면 0으로 생성"""
    for col in PARTICIPATION_ORDERED:
        if col not in df.columns:
            df[col] = 0
    return df

# ===== 노후희망활동 코드 표준화 =====
def map_노후희망활동_2023(x):
    """
    2023의 노후희망활동_1순위코드 -> 표준 '노후희망활동코드'
    규칙: (7,8)->7, 2->1, 그 후 {3:2,4:3,5:4,6:5,7:6}로 재인덱싱
    최종 맵: 1->1, 2->1, 3->2, 4->3, 5->4, 6->5, 7->6, 8->6
    """
    if pd.isna(x): return x
    try: x = int(x)
    except: return x
    if x in (7, 8): x = 7
    if x == 2: x = 1
    mapping = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6, 8:6}
    return mapping.get(x, x)

def map_노후희망활동_2021(x):
    """
    2021: (2,1)->1, 이후 1,1,3,4,5,6,7 -> 1,1,2,3,4,5,6
    최종 맵: 1->1, 2->1, 3->2, 4->3, 5->4, 6->5, 7->6
    """
    if pd.isna(x): return x
    try: x = int(x)
    except: return x
    if x in (1,2): x = 1
    mapping = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6}
    return mapping.get(x, x)




# ===== 메인 표준화 함수 =====
def standardize_by_year(df, year):
    df = df.copy()
    df["년도"] = int(year)

    # ---------- 재학상태코드: 2015/2017/2019 2<->3 ---------- 
    if year in (2015, 2017, 2019):
        if "재학상태코드" in df.columns:
            mapping = {2:3, 3:2}
            df = apply_map(df, "재학상태코드", mapping)

    # ---------- 혼인상태코드 ----------
    if year in (2023, 2021, 2019):
        df = apply_map(df, "혼인상태코드", {5:2})
    elif year == 2017:
        mapping = {1:1, 2:2, 3:1, 4:3, 5:4, 6:5}
        df = apply_map(df, "혼인상태코드", mapping)

    # ---------- 혼인신고여부: NaN -> 0 ----------
    df = fillna_with(df, "혼인신고여부", 0)

    # ---------- 사회보험료부담인식_* ----------
    for col in ["사회보험료부담인식_건강보험부담정도코드",
                "사회보험료부담인식_국민연금부담정도코드",
                "사회보험료부담인식_고용보험부담정도코드"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)
            df[col] = ensure_int_codes(df[col])

    # ---------- 향후추가대상복지서비스_* ----------
    if year in (2021, 2023):
        final_map = {1:1,2:2,3:3,4:4,5:5,6:6,7:6,8:8,9:7,10:8}
        for c in ["향후추가대상복지서비스_1순위코드","향후추가대상복지서비스_2순위코드","향후추가대상복지서비스_3순위코드"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").map(final_map).fillna(df[c])
                df[c] = ensure_int_codes(df[c])

    # ---------- 장애인유대관계대상_* ----------
    for c in ["장애인유대관계대상_1순위코드","장애인유대관계대상_2순위코드","장애인유대관계대상_3순위코드"]:
        df = fillna_with(df, c, 0)

    # ---------- 우선실시장애인복지사업_* ----------
    if year in (2021, 2023):
        final_map = {1:1,2:2,3:2,4:3,5:4,6:4,7:5,8:6,9:7,10:8,11:9}
        for c in ["우선실시장애인복지사업_1순위코드","우선실시장애인복지사업_2순위코드","우선실시장애인복지사업_3순위코드"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").map(final_map).fillna(df[c])
                df[c] = ensure_int_codes(df[c])

    # ---------- 노후준비방법 ----------
    def _map_노후준비(x):
        if pd.isna(x): return x
        try: x = int(x)
        except: return x
        if x == 4: 
            x = 9
        remap = {5:4, 6:5, 7:6, 8:7, 9:8}
        return remap.get(x, x)

    for base in ["노후준비방법_주된방법코드","노후준비방법_부수적방법코드"]:
        if year in (2021, 2023) and base in df.columns:
            df[base] = df[base].apply(_map_노후준비)
            df[base] = ensure_int_codes(df[base])
        df = fillna_with(df, base, 0)

    # ---------- 노후희망활동 ----------
    if year == 2023 and "노후희망활동_1순위코드" in df.columns:
        df["노후희망활동코드"] = df["노후희망활동_1순위코드"].apply(map_노후희망활동_2023).astype("Int64")
    if year == 2021 and "노후희망활동코드" in df.columns:
        df["노후희망활동코드"] = df["노후희망활동코드"].apply(map_노후희망활동_2021).astype("Int64")

    # ---------- 현재자녀동거여부 ----------
    df = fillna_with(df, "현재자녀동거여부", 0)

    # ---------- 현재자녀동거사유 ----------
    if year in (2015, 2017, 2019):
        swap_map = {4:5, 5:4}
        for c in ["현재자녀동거사유_1순위코드","현재자녀동거사유_2순위코드","현재자녀동거사유_3순위코드"]:
            df = apply_map(df, c, swap_map)
    for c in ["현재자녀동거사유_1순위코드","현재자녀동거사유_2순위코드","현재자녀동거사유_3순위코드"]:
        df = fillna_with(df, c, 0)

    # ---------- 현재자녀비동거사유 ----------
    if year in (2015, 2017, 2019):
        map_non = {5:3, 6:5}
        for c in ["현재자녀비동거사유_1순위코드","현재자녀비동거사유_2순위코드","현재자녀비동거사유_3순위코드"]:
            df = apply_map(df, c, map_non)
    for c in ["현재자녀비동거사유_1순위코드","현재자녀비동거사유_2순위코드","현재자녀비동거사유_3순위코드"]:
        df = fillna_with(df, c, 0)

    # ---------- 향후자녀동거의향여부 ----------
    df = fillna_with(df, "향후자녀동거의향여부", 0)

    # ---------- 기부여부 ----------
    if year == 2015 and "기부여부" in df.columns:
        df = apply_map(df, "기부여부", {1:1,2:1,3:1,4:2})

    # ---------- 미기부사유코드 ----------
    if year in (2015, 2017, 2019):
        df = apply_map(df, "미기부사유코드", {1:2, 2:1})
    df = fillna_with(df, "미기부사유코드", 0)

    # ---------- 단체참여 ----------
    if year in (2015, 2017):
        rank_cols = ["참여단체_1순위코드","참여단체_2순위코드","참여단체_3순위코드"]
        df = build_participation_dummies_from_ranks(df, rank_cols)
    else:
        df = ensure_participation_binary(df)

    # ---------- 기부희망분야 ----------
    if year in (2015, 2017, 2019):
        wish_map = {3:7, 7:4, 4:5, 5:6, 6:3}
        for c in ["기부희망분야_1순위코드","기부희망분야_2순위코드","기부희망분야_3순위코드"]:
            df = apply_map(df, c, wish_map)
    for c in ["기부희망분야_1순위코드","기부희망분야_2순위코드","기부희망분야_3순위코드"]:
        df = fillna_with(df, c, 0)

    # ---------- 기부내용_물품횟수 ----------
    if year == 2015 and "기부내용_물품횟수" not in df.columns:
        parts = [
            "기부내용_대상자직접후원_물품횟수",
            "기부내용_언론기관후원_물품횟수",
            "기부내용_물품후원단체후원_물품횟수",
            "기부내용_종교단체후원_물품횟수",
            "기부내용_직장기업후원_물품횟수",
            "기부내용_기타_물품횟수",
        ]
        df = sum_cols(df, "기부내용_물품횟수", parts)

    # ---------- 신문 구독(2015) ----------
    for c in ["신문_일반신문구독여부", "신문_인터넷신문구독여부"]:
        if year == 2015 and c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            s = s.where(~s.isin([1,2,3,4]), 1)
            s = s.where(s != 5, 2)
            df[c] = s.astype("Int64")

    # ---------- 레저시설 ----------
    if year == 2023 and "레저시설_이용여부코드" in df.columns:
        df.rename(columns={"레저시설_이용여부코드": "레저시설이용횟수"}, inplace=True)

    # ---------- 여가활동 동반자코드(2015) ----------
    if year == 2015:
        if "주중여가활동_동반자코드" not in df.columns and "여가활동_동반자코드" in df.columns:
            df["주중여가활동_동반자코드"] = df["여가활동_동반자코드"]
        if "주말여가활동_동반자코드" not in df.columns and "여가활동_동반자코드" in df.columns:
            df["주말여가활동_동반자코드"] = df["여가활동_동반자코드"]

    # ---------- 긴축소비지출항목_* ----------
    if year in (2021, 2023):
        final_map = {1:1,2:2,3:3,4:4,5:5,6:6,7:10,8:7,9:8,10:9,11:10}
        for c in ["긴축소비지출항목_1순위코드","긴축소비지출항목_2순위코드","긴축소비지출항목_3순위코드"]:
            if c in df.columns:
                s = pd.to_numeric(df[c], errors="coerce").map(final_map).fillna(df[c])
                df[c] = pd.Series(s, index=df.index, dtype="Int64")
        for c in ["긴축소비지출항목_1순위코드","긴축소비지출항목_2순위코드","긴축소비지출항목_3순위코드"]:
            df = fillna_with(df, c, 0)
    else:
        for c in ["긴축소비지출항목_1순위코드","긴축소비지출항목_2순위코드","긴축소비지출항목_3순위코드"]:
            df = fillna_with(df, c, 0)

    # ---------- 직업선택요인_* (오류 수정됨) ----------
    if year in (2015, 2017):
        for c in ["직업선택요인_1순위코드","직업선택요인_2순위코드","직업선택요인_3순위코드"]:
            if c in df.columns:
                s = pd.to_numeric(df[c], errors="coerce")
                s_new = np.where(s >= 7, s + 1, s)
                df[c] = pd.Series(s_new, index=df.index, dtype="Int64")

    # ---------- 청년선호직장코드 ----------
    df = fillna_with(df, "청년선호직장코드", 0)
    if year in (2015, 2017) and "청년선호직장코드" in df.columns:
        mapping = {7:4, 4:5, 5:6, 6:7}
        df = apply_map(df, "청년선호직장코드", mapping)

    # ---------- 여성취업시기코드 ----------
    if year in (2015, 2017) and "여성취업시기코드" in df.columns:
        mapping = {1:2, 2:3, 3:4, 4:5, 5:1}
        df = apply_map(df, "여성취업시기코드", mapping)

    # ---------- 여성취업장애요인_* ----------
    if year == 2017:
        for c in ["여성취업장애요인_1순위코드","여성취업장애요인_2순위코드","여성취업장애요인_3순위코드"]:
            if c in df.columns:
                s = pd.to_numeric(df[c], errors="coerce").replace({2:9, 4:9})
                compress_map = {1:1, 3:2, 4:3, 5:4, 6:5, 7:6, 8:7, 9:8, 10:9}
                s = s.map(compress_map).fillna(s)
                df[c] = pd.Series(s, index=df.index, dtype="Int64")
    if year == 2015:
        final_map = {1:1, 2:7, 3:2, 4:3, 5:4, 6:5, 7:6, 8:7, 9:8}
        for c in ["여성취업장애요인_1순위코드","여성취업장애요인_2순위코드","여성취업장애요인_3순위코드"]:
            if c in df.columns:
                s = pd.to_numeric(df[c], errors="coerce").replace({2:8})
                df[c] = pd.Series(s.map(final_map).fillna(s), index=df.index, dtype="Int64")

    # ---------- 고용안정성코드 ----------
    df = fillna_with(df, "고용안정성코드", 0)

    # ---------- 직장산업대분류코드 ----------
    if year in (2021, 2023):
        df = uppercase_col(df, "직장산업대분류코드")

    # ---------- 독서 ----------
    if year == 2023:
        sum_cols(df, "독서_잡지류권수", [
            "독서_잡지류_종이책권수", "독서_잡지류_전자책권수", "독서_잡지류_오디오북권수"
        ])
        sum_cols(df, "독서_교양서적권수", [
            "독서_교양서적_종이책권수", "독서_교양서적_전자책권수", "독서_오디오북_교양서적권수"
        ])
        sum_cols(df, "독서_직업직무관련서적권수", [
            "독서_직업직무관련서적_종이책권수","독서_직업직무관련서적_전자책권수","독서_직업직무관련서적_오디오북권수"
        ])
        sum_cols(df, "독서_생활취미정보서적권수", [
            "독서_생활취미정보서적_종이책권수","독서_생활취미정보서적_전자책권수","독서_생활취미정보서적_오디오북권수"
        ])
        sum_cols(df, "독서_기타권수", [
            "독서_기타_종이책권수","독서_기타_전자책권수","독서_기타_오디오북권수"
        ])

    return df


In [ ]:
import pandas as pd

def load_csv(file):
    try:
        return pd.read_csv(file, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        try:
            return pd.read_csv(file, encoding='cp949', low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(file, encoding='euc-kr', low_memory=False)

df_2015 = load_csv('2015_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2017 = load_csv('2017_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2019 = load_csv('2019_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2021 = load_csv('2021_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2023 = load_csv('2023_복지_사회참여_문화와여가_소득과소비_노동.csv')


# 연도별 표준화
df2015_std = standardize_by_year(df_2015, 2015)
df2017_std = standardize_by_year(df_2017, 2017)
df2019_std = standardize_by_year(df_2019, 2019)
df2021_std = standardize_by_year(df_2021, 2021)
df2023_std = standardize_by_year(df_2023, 2023)

# 공통 스키마로 세로 결합(outer join식으로 컬럼 union)
df_merged = pd.concat([df2015_std, df2017_std, df2019_std, df2021_std, df2023_std],
                      axis=0, ignore_index=True, sort=False)

# 필요 시 컬럼 순서 정리(예: 핵심 식별자 → 년도 → 나머지)
prefer_first = ["가구일련번호","가구원번호","년도"]
cols = prefer_first + [c for c in df_merged.columns if c not in prefer_first]
df_merged = df_merged.reindex(columns=cols)

# 저장
df_merged.to_csv("df_merged_preprocessed.csv", index=False, encoding="utf-8-sig")
log("통합 전처리 완료: df_merged_preprocessed.csv")

[INFO] 통합 전처리 완료: df_merged_preprocessed.csv
